# Trabalho 2: Algoritmos Genéticos — Problema do Caixeiro Viajante (TSP)
**Disciplina:** Inteligência Artificial (IA) — BCC FURB  
**Alunos:** [Inserir nomes dos integrantes do grupo]

### Objetivo
Implementar um algoritmo genético de busca local para encontrar o trajeto de menor distância que visita $n = 20$ cidades e retorna à cidade inicial.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

### 1. Modelagem do Problema e Função de Aptidão (Fitness)

In [ ]:
class Cidade:
    """Representa uma cidade com coordenadas X, Y e identificador id de 1 a 20."""
    def __init__(self, X: float, Y: float, id: int = 0):
        self.X = float(X)
        self.Y = float(Y)
        self.id = int(id)

    def __repr__(self):
        return str(self.id)

def calcularAptidao(cromossomo):
    """
    Calcula a distância euclidiana total percorrida pelo caixeiro viajante,
    fechando o circuito de volta à cidade inicial.
    """
    distancia_total = 0
    n = len(cromossomo)
    for i in range(n):
        atual = cromossomo[i]
        proxima = cromossomo[(i + 1) % n]
        distancia_total += math.sqrt(
            math.pow(proxima.X - atual.X, 2) +
            math.pow(proxima.Y - atual.Y, 2)
        )
    return distancia_total

### 2. Operadores Genéticos (Roleta Linear, Cycle Crossover e Mutação Swap)

In [ ]:
def roleta(melhores_10, pesos):
    """Sorteia um cromossomo dos 10 melhores baseado em pesos lineares (10, 9, ..., 1)."""
    soma_pesos = sum(pesos)
    sorteio = np.random.uniform(0, soma_pesos)
    acumulado = 0
    for i, peso in enumerate(pesos):
        acumulado += peso
        if sorteio <= acumulado:
            return melhores_10[i][0]
    return melhores_10[-1][0]

def cycle_crossover(pai1, pai2):
    """Recombinação Cycle Crossover (CX) conforme passos 1 a 4 do trabalho."""
    filho1 = list(pai1)
    filho2 = list(pai2)
    n = len(filho1)

    pos = np.random.randint(0, n)
    if filho1[pos].id == filho2[pos].id:
        return filho1, filho2

    # Passo 1: troca no local sorteado
    filho1[pos], filho2[pos] = filho2[pos], filho1[pos]

    # Passos 2 a 4: resolução das duplicatas em cadeia
    while True:
        posicoes_vistas = {}
        pos_duplicada = None
        for idx, cidade in enumerate(filho1):
            if cidade.id in posicoes_vistas:
                pos_original = posicoes_vistas[cidade.id]
                pos_duplicada = pos_original if idx == pos else idx
                break
            posicoes_vistas[cidade.id] = idx

        if pos_duplicada is None:
            break

        pos = pos_duplicada
        filho1[pos], filho2[pos] = filho2[pos], filho1[pos]

    return filho1, filho2

def mutacao(cromossomo):
    """Mutação por troca aleatória de dois genes em cada descendente."""
    filho_mutado = list(cromossomo)
    n = len(filho_mutado)
    i, j = np.random.choice(n, size=2, replace=False)
    filho_mutado[i], filho_mutado[j] = filho_mutado[j], filho_mutado[i]
    return filho_mutado

### 3. Carregamento dos Dados das Cidades e Inicialização da População

In [ ]:
# Carrega coordenadas do arquivo cidades.mat
x, y = np.loadtxt("cidades.mat")
cidades = [Cidade(x[i], y[i], id=i + 1) for i in range(len(x))]

TAMANHO_POPULACAO = 20
NUM_GERACOES = 10000
pesos_lineares = [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]

# Inicializa população inicial com 20 permutações aleatórias
populacao = [list(np.random.permutation(cidades)) for _ in range(TAMANHO_POPULACAO)]
populacao_inicial = [list(c) for c in populacao]

### 4. Execução do Algoritmo Genético (10.000 Gerações)

In [ ]:
melhor_custo_global = float("inf")
melhor_solucao_global = None
historico_melhor_custo = []

for geracao in range(NUM_GERACOES):
    # Avaliação e ordenação crescente por aptidão
    avaliados = [(c, calcularAptidao(c)) for c in populacao]
    avaliados.sort(key=lambda ind: ind[1])

    melhor_atual, custo_atual = avaliados[0]
    historico_melhor_custo.append(custo_atual)
    if custo_atual < melhor_custo_global:
        melhor_custo_global = custo_atual
        melhor_solucao_global = list(melhor_atual)

    if (geracao + 1) % 2000 == 0 or geracao == 0:
        print(f"Geração {geracao + 1:5d}/{NUM_GERACOES} | Menor Distância: {custo_atual:.5f} | Melhor Global: {melhor_custo_global:.5f}")

    # Mantém os 10 melhores indivíduos
    dez_melhores = avaliados[:10]

    # Sorteia 5 pares ordenados de pais e gera novos filhos
    novos_filhos = []
    for _ in range(5):
        pai1 = roleta(dez_melhores, pesos_lineares)
        pai2 = roleta(dez_melhores, pesos_lineares)
        while pai1 is pai2:
            pai2 = roleta(dez_melhores, pesos_lineares)

        filho1, filho2 = cycle_crossover(pai1, pai2)
        novos_filhos.append(mutacao(filho1))
        novos_filhos.append(mutacao(filho2))

    # Recomposição da população (10 mantidos + 10 novos)
    populacao = [ind[0] for ind in dez_melhores] + novos_filhos

avaliados_final = [(c, calcularAptidao(c)) for c in populacao]
avaliados_final.sort(key=lambda ind: ind[1])

### 5. Exibição dos Resultados Exigidos

In [ ]:
print("=" * 70)
print("                   RESULTADOS DO ALGORITMO GENÉTICO")
print("=" * 70)
print(f"• Tamanho da População : {TAMANHO_POPULACAO}")
print(f"• Número de Cidades    : {len(cidades)}")
print(f"• Melhor Custo         : {melhor_custo_global:.5f}")
print(f"• Melhor Solução       : {[c.id for c in melhor_solucao_global]}")

print("\n• População Inicial (Amostra dos 20 cromossomos da Geração 0):")
for i, ind in enumerate(populacao_inicial):
    print(f"  Indivíduo {i + 1:02d}: {[c.id for c in ind]}")

print("\n• População Final (20 cromossomos e custos após 10.000 gerações):")
for i, (ind, custo) in enumerate(avaliados_final):
    print(f"  Indivíduo {i + 1:02d} (Custo: {custo:.5f}): {[c.id for c in ind]}")
print("=" * 70)

### 6. Visualização Gráfica (`Plot`)

In [ ]:
caminho = list(melhor_solucao_global) + [melhor_solucao_global[0]]
xs = [c.X for c in caminho]
ys = [c.Y for c in caminho]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico 1: Trajeto das Cidades
ax1.plot(xs, ys, color='royalblue', linestyle='-', linewidth=2, label='Trajeto Fechado', zorder=1)
ax1.scatter([c.X for c in melhor_solucao_global], [c.Y for c in melhor_solucao_global], 
            color='crimson', s=90, edgecolors='black', label='Cidades', zorder=2)
ax1.scatter(melhor_solucao_global[0].X, melhor_solucao_global[0].Y, 
            color='limegreen', s=140, edgecolors='black', label=f'Início (Cidade {melhor_solucao_global[0].id})', zorder=3)

for c in melhor_solucao_global:
    ax1.annotate(f" {c.id}", (c.X, c.Y), fontsize=10, fontweight='bold', color='navy')

ax1.set_title(f"Melhor Caminho do Caixeiro Viajante (TSP)\nDistância Total: {melhor_custo_global:.5f}", fontsize=12, fontweight='bold')
ax1.set_xlabel("Coordenada X")
ax1.set_ylabel("Coordenada Y")
ax1.set_xlim(-0.05, 1.05)
ax1.set_ylim(-0.05, 1.05)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper right')

# Gráfico 2: Curva de Convergência
ax2.plot(historico_melhor_custo, color='darkgreen', linewidth=1.5)
ax2.set_title("Evolução da Aptidão ao Longo das Gerações", fontsize=12, fontweight='bold')
ax2.set_xlabel("Geração")
ax2.set_ylabel("Distância Total (Custo)")
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("melhor_caminho.png", dpi=300)
plt.show()